> **Note:** This notebook continues from `01_cleaningencodeEDAdata_SleepKan.ipynb`. Since notebooks don't share memory across separate Colab sessions, we reload the cleaned & encoded dataset here before continuing.

In [ ]:
import pandas as pd

# Reload the encoded dataset produced by 01_cleaningencodeEDAdata_SleepKan.ipynb
path = '/content/drive/MyDrive/datasetProjectSleep/encoded_sleep_data_ready.csv'
final_data_for_csv = pd.read_csv(path)
print("Dataset shape:", final_data_for_csv.shape)
final_data_for_csv.head()

### 5. Insight Analysis สำหรับ Dashboard (Correlation & Group Analysis)
วิเคราะห์หา insight ที่มีนัยสำคัญก่อนนำไปสร้าง Dashboard ใน Power BI

In [50]:
# 6.1 Correlation กับ Sleep Quality และ Screen Time Affects Sleep
corr = final_data_for_csv.corr()

print("=== Correlation กับ Sleep Quality ===")
print(corr['Sleep Quality_enc'].sort_values(ascending=False))
print()
print("=== Correlation กับ Screen Time Affects Sleep ===")
print(corr['Screen Time Affects Sleep'].sort_values(ascending=False))

=== Correlation กับ Sleep Quality ===
Sleep Quality_enc            1.000000
Sleep Hours                  0.637468
Feel Rested_enc              0.046600
Age                          0.004824
Use Before Sleep_enc        -0.003252
Wellness Apps_enc           -0.019328
Anxiety/Low Mood_enc        -0.039021
Stress Level                -0.498760
Daily Screen Time           -0.518185
Screen Time Affects Sleep   -0.555869
Name: Sleep Quality_enc, dtype: float64

=== Correlation กับ Screen Time Affects Sleep ===
Screen Time Affects Sleep    1.000000
Daily Screen Time            0.586066
Stress Level                 0.533728
Use Before Sleep_enc         0.039713
Wellness Apps_enc            0.017337
Feel Rested_enc              0.012302
Anxiety/Low Mood_enc         0.009873
Age                         -0.029647
Sleep Quality_enc           -0.555869
Sleep Hours                 -0.674748
Name: Screen Time Affects Sleep, dtype: float64


In [51]:
# 6.2 แบ่งกลุ่ม Screen Time และดูผลต่อ Sleep Quality
final_data_for_csv['ScreenTimeBucket'] = pd.cut(
    final_data_for_csv['Daily Screen Time'],
    bins=[0, 4, 7, 10],
    labels=['Low (1-4h)', 'Medium (5-7h)', 'High (8-10h)']
)

screen_time_summary = final_data_for_csv.groupby('ScreenTimeBucket', observed=True).agg(
    avg_sleep_hours=('Sleep Hours', 'mean'),
    pct_good_quality=('Sleep Quality_enc', 'mean'),
    avg_stress=('Stress Level', 'mean'),
    n=('Age', 'count')
)
print("=== สรุปตาม Screen Time Bucket ===")
display(screen_time_summary)

=== สรุปตาม Screen Time Bucket ===


,avg_sleep_hours,pct_good_quality,avg_stress,n
ScreenTimeBucket,,,,
Low (1-4h),8.894366,0.989437,3.605634,284
Medium (5-7h),7.008811,0.651982,4.986784,454
High (8-10h),5.485327,0.354402,6.406321,443


In [52]:
# 6.3 แบ่งกลุ่มอายุ และดูผลต่อ Sleep Quality
final_data_for_csv['AgeGroup'] = pd.cut(
    final_data_for_csv['Age'],
    bins=[17, 25, 35, 45, 51],
    labels=['18-25', '26-35', '36-45', '46-51']
)

age_summary = final_data_for_csv.groupby('AgeGroup', observed=True).agg(
    avg_sleep_hours=('Sleep Hours', 'mean'),
    avg_screen_time=('Daily Screen Time', 'mean'),
    avg_stress=('Stress Level', 'mean'),
    pct_good_quality=('Sleep Quality_enc', 'mean'),
    n=('Age', 'count')
)
print("=== สรุปตามช่วงอายุ (เทียบว่าอายุมีผลน้อยกว่า Screen Time) ===")
display(age_summary)

=== สรุปตามช่วงอายุ (เทียบว่าอายุมีผลน้อยกว่า Screen Time) ===


,avg_sleep_hours,avg_screen_time,avg_stress,pct_good_quality,n
AgeGroup,,,,,
18-25,6.866324,6.629820,5.046272,0.604113,389
26-35,6.854839,6.439516,5.427419,0.649194,248
36-45,6.907473,6.526690,5.106762,0.633452,281
46-51,6.942966,6.391635,5.254753,0.608365,263


In [53]:
# 6.4 สรุป Key Insight เป็นข้อความ (เอาไปใส่ README / Power BI text box)
pct_low = screen_time_summary.loc['Low (1-4h)', 'pct_good_quality'] * 100
pct_high = screen_time_summary.loc['High (8-10h)', 'pct_good_quality'] * 100

print("KEY INSIGHTS")
print(f"1. กลุ่ม Screen Time ต่ำ (1-4 ชม.) มี Sleep Quality ดี {pct_low:.1f}%")
print(f"   ส่วนกลุ่ม Screen Time สูง (8-10 ชม.) มี Sleep Quality ดีแค่ {pct_high:.1f}%")
print(f"2. ช่วงอายุแทบไม่มีผลต่อ Sleep Quality (สังเกตค่า pct_good_quality ใน age_summary ใกล้เคียงกันทุกกลุ่ม)")
print(f"3. Stress Level สัมพันธ์กับ Screen Time (corr = {corr.loc['Stress Level','Daily Screen Time']:.2f})")

KEY INSIGHTS
1. กลุ่ม Screen Time ต่ำ (1-4 ชม.) มี Sleep Quality ดี 98.9%
   ส่วนกลุ่ม Screen Time สูง (8-10 ชม.) มี Sleep Quality ดีแค่ 35.4%
2. ช่วงอายุแทบไม่มีผลต่อ Sleep Quality (สังเกตค่า pct_good_quality ใน age_summary ใกล้เคียงกันทุกกลุ่ม)
3. Stress Level สัมพันธ์กับ Screen Time (corr = 0.49)


### 6. Export ข้อมูลเวอร์ชัน Decoded สำหรับ Power BI
แปลงค่า Encoded (0/1/2) กลับเป็นข้อความที่อ่านรู้เรื่อง เพื่อใช้ทำ Dashboard

In [54]:
# 7.1 Decode ค่ากลับเป็นข้อความ แล้ว export เป็น CSV สำหรับ Power BI
binary_map = {0: 'No', 1: 'Yes'}
rested_map = {0: 'No', 1: 'Sometimes', 2: 'Yes'}
quality_map = {0: 'Bad', 1: 'Good'}
affects_map = {0: 'No', 1: 'Not Sure', 2: 'Yes'}

powerbi_df = pd.DataFrame()
powerbi_df['Age'] = final_data_for_csv['Age']
powerbi_df['Sleep Hours'] = final_data_for_csv['Sleep Hours']
powerbi_df['Daily Screen Time'] = final_data_for_csv['Daily Screen Time']
powerbi_df['Stress Level'] = final_data_for_csv['Stress Level']
powerbi_df['Use Before Sleep'] = final_data_for_csv['Use Before Sleep_enc'].map(binary_map)
powerbi_df['Anxiety/Low Mood'] = final_data_for_csv['Anxiety/Low Mood_enc'].map(binary_map)
powerbi_df['Wellness Apps'] = final_data_for_csv['Wellness Apps_enc'].map(binary_map)
powerbi_df['Feel Rested'] = final_data_for_csv['Feel Rested_enc'].map(rested_map)
powerbi_df['Sleep Quality'] = final_data_for_csv['Sleep Quality_enc'].map(quality_map)
powerbi_df['Screen Time Affects Sleep'] = final_data_for_csv['Screen Time Affects Sleep'].map(affects_map)
powerbi_df['Age Group'] = final_data_for_csv['AgeGroup']
powerbi_df['Screen Time Bucket'] = final_data_for_csv['ScreenTimeBucket']

save_path_powerbi = '/content/drive/MyDrive/datasetProjectSleep/sleep_data_for_powerbi.csv'
powerbi_df.to_csv(save_path_powerbi, index=False)
print(f"บันทึกไฟล์สำหรับ Power BI เรียบร้อยที่: {save_path_powerbi}")
powerbi_df.head()

บันทึกไฟล์สำหรับ Power BI เรียบร้อยที่: /content/drive/MyDrive/datasetProjectSleep/sleep_data_for_powerbi.csv


,Age,Sleep Hours,Daily Screen Time,Stress Level,Use Before Sleep,Anxiety/Low Mood,Wellness Apps,Feel Rested,Sleep Quality,Screen Time Affects Sleep,Age Group,Screen Time Bucket
0,47,4,10,10,Yes,No,No,Yes,Bad,Yes,46-51,High (8-10h)
1,48,4,8,7,No,Yes,Yes,No,Bad,Yes,46-51,High (8-10h)
2,40,7,10,4,No,No,Yes,Sometimes,Bad,Yes,36-45,High (8-10h)
3,26,10,5,1,Yes,Yes,No,Yes,Good,No,26-35,Medium (5-7h)
4,47,5,7,7,Yes,Yes,No,Yes,Good,Yes,46-51,Medium (5-7h)
